# CAMELS: Time Series for the Website
***

**_Autor:_** Chus Casado Rodríguez<br>
**_Fecha:_** 03-08-2026<br>

**Introduction:**<br>
This script combines the daily discharge records with the daily basin meteorology computed from the EMO-1 dataset and expormeteo a time series per gauging station to be plotted in the website.

In [1]:
from pathlib import Path
from tqdm.auto import tqdm
import pandas as pd
import geopandas as gpd

import logging
logger = logging.getLogger(__name__)

from ocab.config import Config
import ocab.variables as vars
from ocab.plots.stations import plot_station_timeseries, create_station_html
from ocab.plots.utils import compute_climatology


## Configuration


In [2]:
cfg = Config('config_CAMELS_v200.yml')

# use this meteo dataset
meteo_ds = 'ROCIO-IBEB' # EMO1

# LSTM model
model = 'bs256_dlr_se3ly_de2ly_2206_104834'

# paths
path_in = cfg.path_dataset / 'preprocessing' / 'timeseries'
path_results = cfg.path_dataset / 'results' / model if model is not None else None
path_web = Path('../../docs')
path_layers = path_web / 'layers'
path_ts = path_web / 'timeseries' / 'stations'
path_plots = path_ts / 'plots'
path_plots.mkdir(exist_ok=True, parents=True)

# point layer
filename = 'stations.geojson'


## Create time series


In [3]:
# load points
points = gpd.read_file(cfg.path_gis / filename).set_index('id')

# add performance
performance_file = path_results / 'performance.geojson'
if performance_file.is_file():
    # read performance values
    performance = gpd.read_file(performance_file)
    performance.rename(columns={'gauge_id': 'id'}, inplace=True)
    performance.set_index('id', inplace=True)
    performance = performance[~performance.index.duplicated(keep='first')]
    performance = performance.loc[performance.index.intersection(points.index)]
    # concatenate to points
    points = pd.concat([points, performance.drop(columns='geometry')], axis=1)
else:
    logger.warning(f'The file {performance_file} does not exist')

In [5]:
cfg.end

Timestamp('2025-09-30 00:00:00')

In [6]:

# process timeseries for each station
for ID in tqdm(points.index, desc='points'):
    
    # observed discharge timeseries
    try:
        observation = pd.read_parquet(path_in / 'discharge' / f'{ID}.parquet')
        observation.columns = ['discharge_cms']
        # compute specific discharge (mm/day)
        observation['discharge_mm'] = observation['discharge_cms'] / points.loc[ID, 'catch_skm'] * 86400 / 1000
    except Exception as e:
        logger.error(f'Loading discharge timeseries for station {ID:04d}: {e}')
        continue

    # simulated discharge timeseries
    try:
        simulation_file = path_results / f'{ID}.parquet'
        if simulation_file.is_file():
            simulation = pd.read_parquet(simulation_file)
            cols = [col for col in simulation if col.endswith('sim')]
            simulation = simulation[cols]
        else:
            simulation = pd.DataFrame
    except Exception as e:
        logger.error(f'Loading simulated discharge timeseries for station {ID:04d}: {e}')

    # meteo timeseries
    try:
        try:
            meteo = pd.read_parquet(path_in / 'meteo' / meteo_ds / f'{ID}.parquet').loc[ID]
        except:
            meteo = pd.read_parquet(path_in / 'meteo' / 'EMO1' / f'{ID}.parquet').loc[ID]
        meteo.rename(columns=vars.RENAME, inplace=True, errors='ignore')
        # ensure average temperature exists
        if 'temp_degC' not in meteo.columns:
            meteo['temp_degC'] = meteo[['temp_max_degC', 'temp_min_degC']].mean(axis=1)
        # correct dates
        if meteo_ds == 'EMO1':
            meteo.index = meteo.index.date - pd.Timedelta(days=1)
        meteo.index.name = 'date'
        meteo.index = pd.to_datetime(meteo.index)
    except Exception as e:
        logger.error(f'Loading meteo timeseries for station {ID}: {e}')
        continue
    
    # merge timeseries
    start = max(cfg.start, meteo.first_valid_index(), observation.first_valid_index() - pd.Timedelta(days=365))
    end = min(cfg.end, meteo.last_valid_index(), observation.last_valid_index())
    ts = pd.concat(
        [observation.loc[start:end], meteo.loc[start:end]], 
        axis=1,
        sort=True
    )
    if simulation_file.is_file():
        ts = pd.concat([ts, simulation], axis=1, sort=True)
    ts = ts[ts.columns.intersection(vars.DECIMALS)].round(vars.DECIMALS)

    # export timeseries
    ts.to_parquet(path_ts / f'{ID:04d}.parquet')

    # compute climatological values
    climatology = compute_climatology(ts)
    points.loc[ID, climatology.index] = climatology.round(0)

    try:
        # extract attributes
        attrs = points.loc[ID]

        # create time series plot
        title = '{0} - {1} - River {2} ({3})'.format(
            ID, 
            attrs['name'].title() if pd.notna(attrs['name']) else '', 
            attrs['river'].title() if pd.notna(attrs['river']) else '', 
            attrs['basin'].title()
        )
        st, en = ts['discharge_mm'].first_valid_index(), ts['discharge_mm'].last_valid_index()
        fig = plot_station_timeseries(
            ts.loc[st:en],
            attrs,
            title=title,
            regime=attrs['regime'],
            save=True
        )
        
        # save plot as HTML
        create_station_html(
            fig,
            path=path_plots / f'{ID}.html', 
            start=ts.index.min().strftime('%Y-%m-%d'), 
            end=ts.index.max().strftime('%Y-%m-%d')
        )
    except Exception as e:
        print(f"The plot for time series {ID} couldn't be created: {e}")

# export updated point layer
points.to_file(path_layers / filename)

points:   0%|          | 0/1116 [00:00<?, ?it/s]